# Stage 8: writing the report

**What this stage does:** writes our final report in the structure of the course template. Every number in the
text is read from the output tables, so the report can never disagree with the results.

Each section of the template has its own cell below, so it's easy to find a section and edit it. The last cells
build the Word, Markdown, HTML and PDF files and check that we are under the 5,000-word limit.

**Reads:** `outputs/tables/*`, `data/processed/*`, `outputs/figures/*` and `report/authors.json`
**Writes:** `report/report.docx`, `report/report.md`, `report/report.html` and the PDF named after the group

In [1]:
%run ./library/lib_00_settings.ipynb
%run ./library/lib_07_report_writer.ipynb
stage_log = get_stage_logger("08_report")

## Loading the results

`ResultNumbers` loads all the output tables once and has a few small helpers that turn results into text.
For example `numbers.value("Test", "density", 3)` gives `"0.263"` and `numbers.three("density", 3)` gives the
Test, ODI and T20I values as "a, b and c".

In [2]:
class ResultNumbers:
    """Every result the report needs, read from the pipeline outputs."""

    def __init__(self) -> None:
        summary_table = pd.read_csv(TABLES_FOLDER / "network_summary.csv").set_index("measure")
        self.summary = {format_name: summary_table[format_name] for format_name in FORMATS}
        self.node_measures = {f: read_table(TABLES_FOLDER / f"node_metrics_{FORMAT_FILE_TAG[f]}.csv") for f in FORMATS}
        self.edges = {f: read_table(PROCESSED_FOLDER / f"edges_{FORMAT_FILE_TAG[f]}.csv") for f in FORMATS}
        self.counts = pd.read_csv(TABLES_FOLDER / "dataset_counts.csv").set_index("format")
        self.file_audit = read_table(INTERIM_FOLDER / "file_audit.csv")
        self.observations = read_table(PROCESSED_FOLDER / "partnership_observations.csv")
        self.scorecard_checks = pd.read_csv(TABLES_FOLDER / "validation_scorecard_checks.csv")
        self.aggregation_checks = pd.read_csv(TABLES_FOLDER / "validation_aggregation_checks.csv")
        self.matches = read_table(PROCESSED_FOLDER / "match_metadata.csv")
        self.manifest = pd.read_csv(INTERIM_FOLDER / "source_manifest.csv")

    def value(self, format_name: str, measure: str, decimals: int = 2) -> str:
        """One network-level value as text. With 0 decimals we add a thousands separator."""
        number = float(self.summary[format_name][measure])
        return f"{number:,.0f}" if decimals == 0 else f"{number:.{decimals}f}"

    def three(self, measure: str, decimals: int = 2) -> str:
        """The Test, ODI and T20I values written as 'a, b and c'."""
        test_value, odi_value, t20_value = (self.value(f, measure, decimals) for f in FORMATS)
        return f"{test_value}, {odi_value} and {t20_value}"

    def table_row(self, label: str, measure: str, decimals: int = 2) -> list[str]:
        """A row for a results table: the label and the value in each format."""
        return [label] + [self.value(f, measure, decimals) for f in FORMATS]

    def leaders(self, format_name: str, measure: str, how_many: int = 3, decimals: int = 0) -> str:
        """The top batters on one measure with their values, e.g. 'JE Root (38), BA Stokes (31), ...'.
        Ties are broken by player ID so the list is the same on every run."""
        top_rows = (self.node_measures[format_name].sort_values([measure, "player_id"], ascending=[False, True])
                    .head(how_many))
        return ", ".join(f"{name} ({value:,.{decimals}f})" for name, value in zip(top_rows["player_name"], top_rows[measure]))

    def top_names(self, format_name: str, measure: str, how_many: int = 3) -> list[str]:
        """Only the names of the top batters on one measure."""
        top_rows = (self.node_measures[format_name].sort_values([measure, "player_id"], ascending=[False, True])
                    .head(how_many))
        return list(top_rows["player_name"])

    def heaviest_pair(self, format_name: str) -> str:
        """The pair with the most partnership runs, written out for the text."""
        top_edge = self.edges[format_name].nlargest(1, "weight").iloc[0]
        return (f"{top_edge.player_1_name} and {top_edge.player_2_name} "
                f"({top_edge.weight:,} runs in {top_edge.partnership_count} partnerships)")

    def community_members(self, format_name: str, community_number: int, how_many: int = 4) -> str:
        """The strongest few members of one community."""
        members = self.node_measures[format_name]
        members = members[members["community"] == community_number].nlargest(how_many, "strength")
        return ", ".join(members["player_name"])

    def community_size(self, format_name: str, community_number: int) -> int:
        measures = self.node_measures[format_name]
        return int((measures["community"] == community_number).sum())

    def main_core(self, format_name: str) -> pd.DataFrame:
        """The batters in the highest k-core of a format."""
        measures = self.node_measures[format_name]
        return measures[measures["core_number"] == measures["core_number"].max()]


numbers = ResultNumbers()
report_details = json.loads((REPORT_FOLDER / "authors.json").read_text(encoding="utf-8"))

## A few facts we use more than once

In [3]:
total_files = len(numbers.file_audit)
included_files = int(numbers.file_audit["included"].sum())
total_observations = len(numbers.observations)
total_innings = int(numbers.counts["england_innings"].sum())
total_runs = int(numbers.counts["partnership_runs"].sum())
total_deliveries = int(numbers.counts["deliveries"].sum())
matches_without_england_batting = int((numbers.matches["n_innings_england"] == 0).sum())
first_match_day, last_match_day = numbers.matches["start_date"].min(), numbers.matches["start_date"].max()
schema_versions = sorted(numbers.manifest["data_version"].astype(str).unique())

players_per_format = [set(numbers.node_measures[f]["player_id"]) for f in FORMATS]
players_in_all_formats = len(players_per_format[0] & players_per_format[1] & players_per_format[2])
distinct_players = len(players_per_format[0] | players_per_format[1] | players_per_format[2])


def rank_correlation_with_innings(format_name: str, measure: str) -> float:
    """Spearman rank correlation between a measure and the number of innings batted."""
    measures = numbers.node_measures[format_name]
    return measures[measure].corr(measures["innings_batted"], method="spearman")


strength_vs_innings = {f: rank_correlation_with_innings(f, "strength") for f in FORMATS}
core_strength_share = {f: numbers.main_core(f)["strength"].sum() / numbers.node_measures[f]["strength"].sum()
                       for f in FORMATS}
core_median_innings = {f: numbers.main_core(f)["innings_batted"].median() for f in FORMATS}
periphery_median_innings = {f: numbers.node_measures[f][numbers.node_measures[f]["core_number"]
                                                        < numbers.node_measures[f]["core_number"].max()]["innings_batted"].median()
                            for f in FORMATS}
scorecard_total, scorecard_passed = len(numbers.scorecard_checks), int(numbers.scorecard_checks["passed"].sum())
test_measures = numbers.node_measures["Test"]


def percent(value: float) -> str:
    return f"{value:.0%}"


def average_clustering_of(format_name: str, player_names: list[str]) -> float:
    """Average local clustering of a few named batters (used for the bridge players)."""
    measures = numbers.node_measures[format_name]
    return float(measures[measures["player_name"].isin(player_names)]["clustering"].mean())


report_blocks: list[tuple] = []
def heading(text): report_blocks.append(("h", text))
def paragraph(text): report_blocks.append(("p", text))

## 1. Introduction

In [4]:
heading("Introduction")
paragraph(
    "Network analysis looks at a system through the ties between its parts. Team sport is a natural fit for "
    "this view, and cricket batting has a very clear tie: the partnership. Two batters are always at the "
    "crease together and build the score as a pair. When a wicket falls, one of them is replaced and a new "
    "partnership starts. Over many matches these pairs form a network of who batted with whom, and how many "
    "runs they added together.")
paragraph(
    "In this project we study the batting partnerships of the England men's team from "
    f"{first_match_day:%d %B %Y} to {last_match_day:%d %B %Y}. England play three international formats: "
    "Test matches (up to five days, two innings per side), One Day Internationals (ODIs, 50 overs per side) "
    "and Twenty20 Internationals (T20Is, 20 overs per side). Since the formats are played differently and "
    "often by different players, we build one network for each format and analyse all three in the same way. "
    "The networks show how widely each batter is connected, where the runs are concentrated, which groups of "
    "batters often bat together and who links these groups.")

## 2. Problem and Motivation

In [5]:
heading("Problem and Motivation")
paragraph(
    "Cricket statistics usually describe single players, for example batting averages, or single "
    "partnerships, for example record stands. We wanted to see the whole batting group at once. A batting "
    "order is not a fixed set of pairs: openers meet the middle order, the middle order meets the tail, and "
    "players come and go over the years. Looking at the structure of these ties shows how much a batting side "
    "relies on a small core, how the top order and the tail are linked, and which players connect different "
    "parts of the squad. Using the same design on three formats also lets us compare them directly.")
paragraph("Research questions. We ask seven questions, one for each part of the analysis.")
for question in [
    "RQ1. What does each network look like overall, and how cohesive is it?",
    "RQ2. Which batters are the most central?",
    "RQ3. How clustered are the partnerships, and how redundant are a batter's ties?",
    "RQ4. Which batters have the highest weighted degree, that is, the most partnership runs?",
    "RQ5. Which communities of batters can we find?",
    "RQ6. Which batters act as bridges (betweenness centrality)?",
    "RQ7. Is there a core of batters with a periphery around it?",
]:
    paragraph(question)
paragraph(
    "Our contribution is a step-by-step method that rebuilds every partnership from ball-by-ball data, three "
    "weighted partnership networks (one per format), and a side-by-side comparison of the three networks "
    "using the same set of measures.")

## 3. Datasets

In [6]:
heading("Datasets")
paragraph(
    "Source. We used the public ball-by-ball data from Cricsheet (cricsheet.org), in the file "
    f"england_male_json.zip. It holds {total_files} matches played by England men's teams, one JSON file per "
    f"match (Cricsheet format version {', '.join(schema_versions)}). The data is already digital, so nothing "
    "had to be typed in by hand. We keep the zip file unchanged in our project folder, together with a list "
    "of all its files and their checksums.")
paragraph(
    "Boundary of the study. We set the boundary of the network with clear, formal rules (a nominalist "
    "approach). A match is part of the study if it is a men's international match, England is playing, the "
    "format is Test, ODI or T20I, and its first day falls between 1 January 2020 and 29 October 2026. "
    f"{included_files} of the {total_files} matches meet these rules. The others were all played before 2020. "
    f"In {matches_without_england_batting} limited-overs matches England did not get to bat, so these matches "
    "add no partnerships. Table 1 shows the data we worked with.")
report_blocks.append(("table", "Table 1. The data used in the study (England batting innings only).",
    ["", "Test", "ODI", "T20I", "Total"],
    [[label] + [f"{int(numbers.counts.loc[f, column]):,}" for f in FORMATS] + [f"{int(numbers.counts[column].sum()):,}"]
     for label, column in [("Matches", "matches"), ("England innings", "england_innings"),
                           ("Balls (deliveries)", "deliveries"), ("Partnerships", "observations"),
                           ("Partnership runs", "partnership_runs")]]
    + [["Nodes (batters)"] + [numbers.value(f, "nodes", 0) for f in FORMATS] + [f"{distinct_players} different"],
       ["Edges (pairs)"] + [numbers.value(f, "edges", 0) for f in FORMATS] + ["-"]]))
paragraph(
    "Finding the partnerships. For every England innings we read the balls in order. Each ball names the "
    "striker and the non-striker, gives the runs (runs.total, which already includes extras such as wides and "
    "no-balls) and lists any wicket. Our code remembers the current pair and adds the runs of each ball to it "
    "once. When a wicket record names one of the two batters, the partnership ends after that ball, so the runs "
    "scored on the wicket ball still go to the pair that was batting. The last partnership ends with the "
    "innings. If two batters are split up and later bat together again, for example after a retired-hurt "
    f"batter comes back, we count two partnerships. In total we found {total_observations:,} partnerships.")
paragraph(
    "The network model. For each format we built one undirected, weighted (valued) and one-mode (monomodal) "
    "network. A node is a batter. An edge joins two batters who shared at least one partnership. The edge "
    "weight is the total number of runs the pair added together across all their partnerships in that format. "
    "Only batters who actually shared the crease become nodes. Across the three formats there are "
    f"{distinct_players} different batters, and {players_in_all_formats} of them appear in all three networks.")
paragraph(
    "Tools. We handled the data in Python with pandas, computed the network measures with the NetworkX "
    "library and drew the figures with Matplotlib. All the work is organised as Jupyter notebooks that run in "
    "a fixed order, from the raw zip file to this report.")

## 4. Validity and Reliability
Validity asks "are we measuring what we intend to measure?", that is, how closely our network represents the
real batting partnerships. Reliability asks "if the study were repeated under the same conditions, would we get
the same results?". We follow these two meanings below.

In [7]:
heading("Validity and Reliability")
paragraph(
    "Validity. We want to measure who batted with whom and how many runs each pair added. Our ties are not "
    "guessed from other information: a tie exists only when Cricsheet records two batters at the crease "
    "together on at least one ball. This keeps edge attribution errors out of the model. To avoid omission "
    "errors, we read every ball of every England innings inside the boundary, and every one of the "
    f"{total_files} files could be read. To avoid commission errors, only batters who actually shared a "
    "partnership became nodes. Players who were only in the squad, and the batting of the other team, are not "
    "part of the networks. The edge weight follows the usual cricket meaning of a partnership: the total the "
    "team added while the two batters were together, extras included. Penalty runs that are given to a whole "
    "innings and not to a single ball are not given to any pair. Players are identified by their fixed "
    "Cricsheet ID, so the same player is always the same node, and two different players never share one.")
paragraph(
    "Reliability. The study is built on objective public records and every measure comes from a formula or an "
    "algorithm, so another group starting from the same zip file would get the same networks and the same "
    "numbers. The rules for building the data are written down and fixed in advance: the same boundary for "
    "every match, one network per format, and edge weights that are always the sum of the partnership runs. "
    "The one step that uses random numbers, Louvain community detection (and the random graphs we compare "
    "against), runs with a fixed seed. Because Cricsheet is a secondary source, we also checked that the files "
    f"are consistent with themselves. All files use the same format version. In all {total_innings} innings the "
    f"partnership runs add up exactly to the runs of all the balls ({total_runs:,} runs from "
    f"{total_deliveries:,} balls). Adding up each innings straight from the balls also gives exactly the target "
    f"and the winning margin stored in the same files, in all {scorecard_passed} of {scorecard_total} cases. "
    "Every change of pair in every innings is explained by a wicket, a recorded batter replacement or the end "
    f"of the innings, and {len(numbers.aggregation_checks)} further checks confirm that the networks match the "
    "partnership table exactly. The whole analysis runs again with a single command.")

## 5. Measures and Results

In [8]:
heading("Measures and Results")
paragraph(
    "Edge weights in our networks are tie strengths: more runs means a stronger tie. Measures based on "
    "shortest paths (geodesic distance, betweenness and closeness) instead read edge values as lengths. So we "
    "use the runs only in the measures made for strengths (strength, eigenvector centrality and modularity), "
    "and in all path-based measures every tie counts as one step. Clustering, redundancy and k-cores also look "
    "only at whether a tie exists. Table 2 gives the network-level results and Figure 1 shows the three "
    "networks.")
report_blocks.append(("table", "Table 2. Network-level measures. (W) = uses the edge weights.",
    ["Measure", "Test", "ODI", "T20I"],
    [numbers.table_row("Nodes (batters)", "nodes", 0), numbers.table_row("Edges (pairs)", "edges", 0),
     numbers.table_row("Density", "density", 3), numbers.table_row("Average degree", "degree_mean", 1),
     numbers.table_row("Components", "components", 0),
     numbers.table_row("Connectedness", "connectedness", 2),
     numbers.table_row("Compactness", "compactness", 3),
     numbers.table_row("Mean geodesic distance", "mean_geodesic_distance", 2),
     numbers.table_row("Diameter", "diameter", 0),
     numbers.table_row("Highest degree", "degree_max", 0),
     numbers.table_row("Degree centralisation", "degree_centralisation", 3),
     numbers.table_row("Median edge weight, runs (W)", "edge_weight_median", 1),
     numbers.table_row("Heaviest edge, runs (W)", "edge_weight_max", 0),
     numbers.table_row("Average clustering coefficient", "avg_clustering", 3),
     numbers.table_row("Transitivity", "transitivity", 3),
     numbers.table_row("Average redundancy", "mean_redundancy", 2),
     numbers.table_row("Small-world index (sigma)", "small_world_sigma", 2),
     numbers.table_row("Main k-core (k)", "max_core_k", 0),
     numbers.table_row("Batters in the main k-core", "max_core_nodes", 0),
     numbers.table_row("Louvain communities (W)", "louvain_communities", 0),
     numbers.table_row("Modularity Q (W)", "louvain_modularity", 3)]))
report_blocks.append(("fig", FIGURES_FOLDER / "network_comparison.png",
    "Figure 1. The three batting-partnership networks, drawn the same way. Node size: strength (total "
    "partnership runs). Edge width: runs added by the pair. Colour: Louvain community (numbered by size; grey: "
    "fewer than three players). Batters who share many partners are drawn close together."))

paragraph(
    "Overall shape and cohesion (RQ1). Density is the share of all possible pairs that actually batted "
    "together. Connectedness is the share of pairs of batters that can reach each other through a chain of "
    "partnerships, and compactness also rewards short chains. Each network has about fifty batters "
    f"({numbers.three('nodes', 0)}) and forms one single component, so connectedness is 1 in every format. "
    f"Density is {numbers.three('density', 3)} for Test, ODI and T20I, and the average batter has "
    f"{numbers.three('degree_mean', 1)} partners. Any two batters are on average only about two partnerships "
    f"apart (mean geodesic distance {numbers.three('mean_geodesic_distance')}), and the diameter is "
    f"{numbers.value('Test', 'diameter', 0)} in every format. All three networks are therefore very cohesive. "
    "The degree distribution (Figure 2) is right-skewed: most batters have a handful of partners, while a few "
    "hubs have many. The Test network is the most centralised (degree centralisation "
    f"{numbers.value('Test', 'degree_centralisation', 3)}), because one batter, JE Root, partnered "
    f"{int(test_measures['degree'].max())} of the other {int(numbers.value('Test', 'nodes', 0)) - 1} batters.")
report_blocks.append(("fig", FIGURES_FOLDER / "degree_distributions.png",
    "Figure 2. Degree distribution of each network: how many batters had a given number of different partners. "
    "The dashed line is the average degree."))

paragraph(
    "Central batters (RQ2). Degree centrality counts a batter's different partners. Closeness centrality is "
    "high when a batter is only a few steps from everyone else. Eigenvector centrality is high when a batter "
    "has strong ties to batters who are themselves well connected. Table 3 lists the leading batters. In "
    "Tests, JE Root comes first on every measure. In ODIs, Root leads on degree and eigenvector centrality and "
    "shares the highest closeness with AU Rashid. In T20Is the most connected batter is LS Livingstone, while JC Buttler leads on "
    "eigenvector centrality thanks to strong ties with the other top-order batters.")
report_blocks.append(("table", "Table 3. The three leading batters in each format. Strength and eigenvector "
                               "centrality use the edge weights; the other measures do not.",
    ["Measure", "Test", "ODI", "T20I"],
    [["Degree"] + [numbers.leaders(f, "degree") for f in FORMATS],
     ["Strength (runs)"] + [numbers.leaders(f, "strength") for f in FORMATS],
     ["Closeness"] + [numbers.leaders(f, "closeness", decimals=3) for f in FORMATS],
     ["Eigenvector"] + [numbers.leaders(f, "eigenvector", decimals=3) for f in FORMATS],
     ["Betweenness"] + [numbers.leaders(f, "betweenness", decimals=3) for f in FORMATS]]))

paragraph(
    "Clustering and redundancy (RQ3). The local clustering coefficient of a batter is the share of the batter's "
    "partners who also batted with each other; transitivity is the same idea for the network as a whole. "
    "Redundancy is the average number of ties a partner has to the batter's other partners. Average "
    f"clustering is {numbers.three('avg_clustering', 3)}, more than twice the value of random graphs with "
    f"the same number of nodes and edges ({numbers.three('C_random', 3)}), while the distances stay almost as "
    "short as in those random graphs. This is the small-world effect: the small-world index (sigma) is "
    f"{numbers.three('small_world_sigma')}, well above 1. The average redundancy is "
    f"{numbers.three('mean_redundancy')}, so a typical partner of a batter has also batted with several of the "
    "batter's other partners. The reason is simple: batters who play in the same matches meet each other as "
    "the wickets fall, which closes many triads.")

paragraph(
    "Weighted degree (RQ4). Strength, or weighted degree, is the sum of a batter's edge weights: all the runs "
    "added in the partnerships the batter took part in. The leaders are JE Root in Tests "
    f"({int(test_measures['strength'].max()):,} runs) and in ODIs, and JC Buttler in T20Is (Table 3). The "
    "heaviest pairs are "
    f"{numbers.heaviest_pair('Test')} in Tests, {numbers.heaviest_pair('ODI')} in ODIs and "
    f"{numbers.heaviest_pair('T20I')} in T20Is. Strength rises with the number of innings a batter played "
    f"(rank correlation {strength_vs_innings['Test']:.2f}, {strength_vs_innings['ODI']:.2f} and "
    f"{strength_vs_innings['T20I']:.2f}), so it shows both how productive and how long-serving a batter has "
    "been. Degree and strength can point to different players: in T20Is Livingstone has the most partners, but "
    "Buttler's strength is more than three times higher, because Buttler's runs come from a few very strong ties.")

paragraph(
    "Communities (RQ5). A community is a group of batters with more tie weight inside the group than we would "
    "expect by chance. We found them with the Louvain method, which maximises the modularity Q. We find "
    f"{numbers.three('louvain_communities', 0)} communities (Q = {numbers.three('louvain_modularity', 3)}). "
    "The groups follow two things the method was never told about: batting position and time. In Tests, one "
    f"community holds the middle order ({numbers.community_members('Test', 2)}), one the lower order and the "
    f"tail ({numbers.community_size('Test', 1)} players, such as {numbers.community_members('Test', 1, 3)}), "
    f"and two smaller ones hold the top order of the later years ({numbers.community_members('Test', 3, 3)}) "
    f"and of 2020 to 2021 ({numbers.community_members('Test', 4)}). ODIs split in the same way into a recent top "
    f"order ({numbers.community_members('ODI', 2)}), an earlier opening group "
    f"({numbers.community_members('ODI', 3)}), a middle-order group ({numbers.community_members('ODI', 4)}) "
    f"and a lower-order group of {numbers.community_size('ODI', 1)} players. T20Is have a current top order "
    f"({numbers.community_members('T20I', 2)}), an earlier group ({numbers.community_members('T20I', 3)}) and "
    f"a lower-order group of {numbers.community_size('T20I', 1)} players.")

odi_bridges = numbers.top_names("ODI", "betweenness")
t20_bridges = numbers.top_names("T20I", "betweenness")
paragraph(
    "Bridges (RQ6). Betweenness centrality measures how often a batter lies on the shortest paths between "
    "other batters. A batter with high betweenness links parts of the squad that rarely bat together. In "
    f"Tests this is again Root ({test_measures['betweenness'].max():.3f}, more than twice the next value). In "
    f"ODIs the highest values belong to {', '.join(odi_bridges[:2])} and {odi_bridges[2]}, and in T20Is to "
    f"{', '.join(t20_bridges[:2])} and {t20_bridges[2]}. Most of these are all-rounders and bowlers who bat "
    "lower down: they bat with whoever is left of the top order and with the other bowlers, so they connect "
    "the changing tail with the batting core. Their local clustering is lower than average "
    f"({average_clustering_of('ODI', odi_bridges):.2f} against {numbers.value('ODI', 'avg_clustering')} in ODIs, "
    f"{average_clustering_of('T20I', t20_bridges):.2f} against {numbers.value('T20I', 'avg_clustering')} in T20Is), "
    "which means they sit across structural holes: many of their partners never batted with each "
    "other.")

paragraph(
    "Core and periphery (RQ7). A k-core is a group of batters in which every batter is tied to at least k "
    "others in the group. Peeling away the batters with fewer ties layer by layer leaves the highest k-core, "
    "which we treat as the core; the batters outside it form the periphery. The core is the "
    f"{numbers.value('Test', 'max_core_k', 0)}-core in Tests ({numbers.value('Test', 'max_core_nodes', 0)} "
    f"batters), the {numbers.value('ODI', 'max_core_k', 0)}-core in ODIs ({numbers.value('ODI', 'max_core_nodes', 0)}) "
    f"and the {numbers.value('T20I', 'max_core_k', 0)}-core in T20Is ({numbers.value('T20I', 'max_core_nodes', 0)}). "
    f"These core batters hold {percent(core_strength_share['Test'])}, {percent(core_strength_share['ODI'])} and "
    f"{percent(core_strength_share['T20I'])} of all the strength, and they batted a median of "
    f"{core_median_innings['Test']:.0f}, {core_median_innings['ODI']:.0f} and {core_median_innings['T20I']:.0f} "
    f"innings, against {periphery_median_innings['Test']:g}, {periphery_median_innings['ODI']:g} and "
    f"{periphery_median_innings['T20I']:g} for the batters in the periphery. So every format shows a clear "
    "core-periphery structure. The Test core mixes specialist batters and long-serving bowlers (Root, Stokes and "
    "Brook, but also Broad, Anderson and Leach), because the core rewards batting with many different partners.")

paragraph(
    "Comparing the formats. Since all three networks have a similar size and were measured in the same way, "
    "their structure can be compared directly. The Test network is the densest, the most clustered and the "
    "most centralised, built around one hub. The ODI network has the highest modularity, which means its "
    "communities are the most clearly separated, mostly by era. The T20I network has the sharpest "
    f"core-periphery split: its core of {numbers.value('T20I', 'max_core_nodes', 0)} batters holds "
    f"{percent(core_strength_share['T20I'])} of the strength.")

## 6. Conclusion

In [9]:
heading("Conclusion")
paragraph(
    "All three England batting networks are highly cohesive small worlds. Every batter is linked to every "
    "other through short chains of partnerships, and the partners of a batter very often batted with each "
    "other too. This follows from the way an innings works: batters who play in the same matches meet each "
    "other as wickets fall.")
paragraph(
    "Different measures pick out different kinds of important batters. Strength and eigenvector centrality "
    "point to the top-order and middle-order batters who add most of the runs, such as Root and Buttler. "
    "Betweenness points to the all-rounders and lower-order batters who connect the tail with the core, such "
    "as Rashid, Carse, Livingstone and Jordan. In Tests one batter, Root, leads on every measure, which makes "
    "the Test network the most centralised.")
paragraph(
    "The communities found by Louvain match batting positions and generations of players, even though the "
    "method only saw who batted with whom. The k-core analysis shows a clear core of long-serving batters "
    "with a periphery of players who batted much less often. Between the formats, Test cricket gives the "
    "most tightly knit and hub-centred network, ODI cricket the most clearly divided into groups, and T20I "
    "cricket the strongest core.")

## 7. Critique

In [10]:
heading("Critique")
paragraph(
    "We believe the project answers all seven research questions. The networks rest on a fully traceable "
    "base: every edge can be followed back to the single balls it came from, and the data checks confirm that "
    "no run is lost or counted twice. Using one design for all three formats made the comparison simple and "
    "fair, and the measures we chose cover the main ideas of the course: cohesion, centrality, clustering and "
    "redundancy, communities and core-periphery structure.")
paragraph(
    "With more time we would like to extend the study in a few directions. Building one network per year "
    "would show how the batting group changes from one generation to the next. A directed version, with "
    "edges pointing to the batter who scored the runs, would show who carried each partnership. Adding match "
    "results would let us ask whether some network shapes go together with winning, and running the same "
    "notebooks on other national teams would show whether the patterns we found are typical for international "
    "cricket.")

## References

In [11]:
heading("References")
for reference in [
    "Blondel, V. D., Guillaume, J.-L., Lambiotte, R., Lefebvre, E. (2008). Fast unfolding of communities in "
    "large networks. Journal of Statistical Mechanics, P10008.",
    "Borgatti, S. P., Everett, M. G., Johnson, J. C. (2018). Analyzing Social Networks (2nd ed.). Sage.",
    "Burt, R. S. (1992). Structural Holes: The Social Structure of Competition. Harvard University Press.",
    "Cricsheet. Ball-by-ball cricket data, england_male_json.zip. https://cricsheet.org/downloads/",
    "Freeman, L. C. (1978). Centrality in social networks: conceptual clarification. Social Networks, 1(3), 215-239.",
    "Newman, M. E. J. (2018). Networks (2nd ed.). Oxford University Press.",
    "Seidman, S. B. (1983). Network structure and minimum degree. Social Networks, 5(3), 269-287.",
    "Watts, D. J., Strogatz, S. H. (1998). Collective dynamics of 'small-world' networks. Nature, 393, 440-442.",
]:
    report_blocks.append(("ref", reference))

## Building the files and checking the word limit

In [12]:
write_word_report(report_blocks, report_details, REPORT_FOLDER / "report.docx")
write_markdown_report(report_blocks, report_details, REPORT_FOLDER / "report.md")
write_html_report(report_blocks, report_details, REPORT_FOLDER / "report.html")

report_word_count = count_report_words(report_blocks, report_details)
stage_log.info("report written: %d words (limit %d), %s", report_word_count, WORD_LIMIT,
               "OK" if report_word_count <= WORD_LIMIT else "TOO LONG")
assert report_word_count <= WORD_LIMIT, "The report is over the 5,000-word limit"

2026-11-19 18:05:23,281 INFO partnerships.08_report: report written: 3370 words (limit 5000), OK


### PDF

The course asks for the file to be named after the group members' surnames in alphabetical order, joined by
"and". The name is set in `report/authors.json`.

In [13]:
pdf_path = REPORT_FOLDER / f"{report_details['pdf_name']}.pdf"
print_html_to_pdf(REPORT_FOLDER / "report.html", pdf_path)
set_pdf_details(pdf_path, report_details)
stage_log.info("saved %s", pdf_path.name)

2026-11-19 18:05:24,741 INFO partnerships.08_report: saved Khan and Musale and Raja Mohamed and Salman.pdf
